# Bundle Migration: Generate & Deploy (Modular)

## Overview

Generate Databricks Asset Bundle and deploy to target workspace using configuration from databricks.yml.

**Prerequisites:**
- ✅ Bundle_03 completed (dashboards exported and transformed)
- ✅ databricks.yml configured with target workspace details

**What this notebook does:**
1. Loads transformed dashboards and permissions
2. Generates complete bundle structure
3. Validates bundle configuration
4. Deploys to target workspace via CLI
5. Verifies deployment

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml databricks-cli "numpy<2" --quiet
dbutils.library.restartPython()

## Cell 1: Import Helpers & Load Config

In [ ]:
import sys
import os
from pathlib import Path

# Dynamically locate helpers directory
print("=== HELPERS PATH RESOLUTION DEBUG ===")

try:
    # In Databricks workspace/job context
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    print(f"Notebook path (from dbutils): {notebook_path}")
    
    # Add /Workspace prefix if not present
    if not notebook_path.startswith('/Workspace'):
        notebook_path = '/Workspace' + notebook_path
        print(f"Added /Workspace prefix: {notebook_path}")
    
    # Get parent directory of Bundle folder
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    print(f"Notebook dir: {notebook_dir}")
    print(f"Bundle parent: {bundle_parent}")
    
    # Verify helpers exists in bundle_parent
    helpers_path = os.path.join(bundle_parent, 'helpers')
    print(f"\nChecking helpers path: {helpers_path}")
    
    if os.path.exists(helpers_path):
        print(f"  Helpers directory EXISTS")
        init_file = os.path.join(helpers_path, '__init__.py')
        if os.path.exists(init_file):
            print(f"  __init__.py FOUND")
            print(f"  Contents: {os.listdir(helpers_path)[:10]}")
        else:
            print(f"  WARNING: No __init__.py found")
    else:
        print(f"  WARNING: Helpers directory does not exist at {helpers_path}")
    
    # Add BUNDLE PARENT to sys.path (not the helpers dir itself)
    sys_path_entry = bundle_parent
    
except Exception as e:
    print(f"Error in workspace context: {e}")
    import traceback
    traceback.print_exc()
    # Fallback for local execution
    sys_path_entry = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f"Using local fallback: {sys_path_entry}")

sys_path_entry = os.path.normpath(sys_path_entry)

# Add bundle parent to sys.path (NOT the helpers directory)
print(f"\nAdding to sys.path: {sys_path_entry}")
if sys_path_entry not in sys.path:
    sys.path.insert(0, sys_path_entry)
    print(f"  Added to sys.path")
else:
    print(f"  Already in sys.path")

print(f"\nsys.path first 3 entries: {sys.path[:3]}")
print("=== END DEBUG ===\n")

from helpers import (
    set_config,
    get_path,
    write_volume_file,
    ensure_directory_exists,
    create_workspace_client,
    list_volume_files,
    load_permissions_from_csv,
    load_schedules_from_csv,
    apply_dashboard_schedules,
    generate_bundle_structure,
    validate_bundle,
    deploy_bundle
)
import pandas as pd

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
# ============================================================================
# CONFIGURATION FROM PARAMETERS (databricks.yml)
# ============================================================================
# This notebook reads configuration from job parameters (widgets)
# All configuration is passed from databricks.yml

print("="*80)
print("LOADING CONFIGURATION FROM PARAMETERS")
print("="*80)

# Read all parameters from widgets
catalog = dbutils.widgets.get('catalog')
volume_base = dbutils.widgets.get('volume_base')
source_workspace_url = dbutils.widgets.get('source_workspace_url')
target_workspace_url = dbutils.widgets.get('target_workspace_url')
exported_path_rel = dbutils.widgets.get('exported_path')
transformed_path_rel = dbutils.widgets.get('transformed_path')
bundles_path_rel = dbutils.widgets.get('bundles_path')
target_parent_path = dbutils.widgets.get('target_parent_path')
warehouse_name = dbutils.widgets.get('warehouse_name')
bundle_name = dbutils.widgets.get('bundle_name')
apply_permissions = dbutils.widgets.get('apply_permissions').lower() == 'true'
permissions_dry_run = dbutils.widgets.get('permissions_dry_run').lower() == 'true'
apply_schedules = dbutils.widgets.get('apply_schedules').lower() == 'true'
schedules_dry_run = dbutils.widgets.get('schedules_dry_run').lower() == 'true'
embed_credentials = dbutils.widgets.get('embed_credentials').lower() == 'true'

# Deployment method: 'sdk_direct' or 'asset_bundle'
deployment_method = dbutils.widgets.get('deployment_method').lower()
dry_run_mode = dbutils.widgets.get('dry_run_mode').lower() == 'true'

# Build full paths
exported_path = f"{volume_base}/{exported_path_rel}"
transformed_path = f"{volume_base}/{transformed_path_rel}"
bundles_path = f"{volume_base}/{bundles_path_rel}"

# Build config dict for helper functions
config = {
    'source': {
        'workspace_url': source_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'target': {
        'workspace_url': target_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'paths': {
        'volume_base': volume_base,
        'exported': exported_path_rel,
        'transformed': transformed_path_rel,
        'bundles': bundles_path_rel,
        'target_parent_path': target_parent_path
    },
    'bundle': {
        'name': bundle_name,
        'embed_credentials': embed_credentials
    },
    'warehouse': {
        'warehouse_name': warehouse_name
    },
    'permissions': {
        'apply': apply_permissions,
        'dry_run': permissions_dry_run
    },
    'schedules': {
        'apply': apply_schedules,
        'dry_run': schedules_dry_run
    }
}

# Cache config for helper functions
set_config(config)

print("\n✅ Configuration loaded from parameters\n")
print(f"   Source: {source_workspace_url}")
print(f"   Target: {target_workspace_url}")
print(f"   Volume base: {volume_base}")
print(f"   Bundle name: {bundle_name}")
print(f"   Parent path: {target_parent_path}")
print(f"   Warehouse: {warehouse_name}")
print(f"   Embed credentials: {embed_credentials}")
print(f"   Apply schedules: {'Yes' if apply_schedules else 'No'}")

# Ensure directories exist
print("\n📁 Ensuring directories exist...")
ensure_directory_exists(bundles_path)
print(f"   ✅ Bundles: {bundles_path}")

## Cell 3: Load Transformed Dashboards & Permissions

In [ ]:
print("\n" + "="*80)
print("LOADING DASHBOARDS & PERMISSIONS")
print("="*80)

# Use paths already defined in config
print(f"\n📂 Loading transformed dashboards from: {transformed_path}")
dashboard_files = list_volume_files(transformed_path, '*.lvdash.json')

if not dashboard_files:
    print("   ❌ No transformed dashboards found!")
    print("   💡 Run Bundle_01 first to export and transform dashboards")
    raise Exception("No dashboards to deploy")

print(f"   ✅ Found {len(dashboard_files)} dashboard(s)")

# Load permissions
print(f"\n🔐 Loading permissions from: {exported_path}")
permissions_map = load_permissions_from_csv(exported_path)
print(f"   ✅ Loaded permissions for {len(permissions_map)} dashboard(s)")

# Build dashboard dict with metadata (keyed by dashboard_id)
dashboards = {}
for file_path in dashboard_files:
    filename = Path(file_path).name
    
    # Extract dashboard ID and name from filename
    # Format: dashboard_{id}_{name}.lvdash.json
    parts = filename.replace('.lvdash.json', '').split('_', 2)
    if len(parts) >= 3:
        dashboard_id = parts[1]
        name = parts[2].replace('_', ' ')
    else:
        dashboard_id = filename
        name = filename
    
    # Dict format expected by generate_bundle_structure
    dashboards[dashboard_id] = {
        'display_name': name,
        'source_path': file_path
    }

# Display summary
df = pd.DataFrame([{'Dashboard': v['display_name'], 'ID': k} for k, v in dashboards.items()])
display(df)

## Cell 4: Deployment Path Selection

In [ ]:
print("\n" + "="*80)
print("DEPLOYMENT PATH SELECTION")
print("="*80)

print(f"\n📦 Selected deployment method: {deployment_method.upper()}")
print(f"🔒 Dry run mode: {'Yes' if dry_run_mode else 'No'}")

if deployment_method == 'sdk_direct':
    print("\n" + "-"*80)
    print("SDK DIRECT DEPLOYMENT PATH")
    print("-"*80)
    print("\nThis will deploy dashboards directly via SDK API calls.")
    print("No CLI or local machine required.")
    
elif deployment_method == 'asset_bundle':
    print("\n" + "-"*80)
    print("ASSET BUNDLE DEPLOYMENT PATH")
    print("-"*80)
    print("\nThis will generate a Databricks Asset Bundle.")
    print("You will need to deploy via CLI after this notebook completes.")
    
else:
    raise ValueError(f"Unknown deployment_method: {deployment_method}. Use 'sdk_direct' or 'asset_bundle'")


In [ ]:
# ============================================================================
# SDK DIRECT DEPLOYMENT
# ============================================================================

# Initialize deployment result variables (used by later cells)
deployed_dashboards = []
failed_dashboards = []
deploy_result = {}

if deployment_method == 'sdk_direct':
    print("\n" + "="*80)
    print("SDK DIRECT DEPLOYMENT")
    print("="*80)
    
    from helpers import deploy_via_sdk, resolve_warehouse, create_target_workspace_client
    
    # Create target workspace client
    print("\n🔌 Connecting to target workspace...")
    target_client = create_target_workspace_client(target_workspace_url)
    print(f"   ✅ Connected to: {target_workspace_url}")
    
    # Resolve warehouse
    print("\n🏭 Resolving warehouse...")
    try:
        warehouse_id = dbutils.widgets.get('warehouse_id')
        if not warehouse_id:
            warehouse_id = resolve_warehouse(target_client, warehouse_name)
        print(f"   ✅ Warehouse ID: {warehouse_id}")
    except Exception as e:
        print(f"   ⚠️ Could not resolve warehouse: {e}")
        warehouse_id = None
    
    # Build deployment packages
    print(f"\n📦 Building deployment packages...")
    from helpers import build_deployment_packages
    
    packages = build_deployment_packages(
        transformed_path=transformed_path,
        permissions_map=permissions_map,
        schedules_map=None  # Schedules applied separately in Cell 19
    )
    print(f"   ✅ Built {len(packages)} package(s)")
    
    # Deploy via SDK
    print(f"\n🚀 Deploying {len(packages)} dashboard(s)...")
    print(f"   Target folder: {target_parent_path}")
    print(f"   Dry run: {dry_run_mode}")
    print()
    
    result = deploy_via_sdk(
        client=target_client,
        packages=packages,
        target_parent_path=target_parent_path,
        warehouse_id=warehouse_id,
        warehouse_name=warehouse_name,
        apply_permissions=apply_permissions,
        apply_schedules=False,  # Schedules applied separately in Cell 19
        embed_credentials=embed_credentials,
        dry_run=dry_run_mode
    )
    
    # Extract results from the actual return structure
    deployed_dashboards = [r for r in result.get('results', []) if r['status'] == 'success']
    failed_dashboards = [r for r in result.get('results', []) if r['status'] == 'error']
    
    print("\n" + "="*80)
    print("SDK DEPLOYMENT SUMMARY")
    print("="*80)
    
    # Use the correct keys from the result dictionary
    total_deployed = result.get('successful', len(deployed_dashboards))
    total_failed = result.get('failed', len(failed_dashboards))
    total_skipped = result.get('skipped', 0)
    
    print(f"\n   ✅ Deployed: {total_deployed}")
    print(f"   ❌ Failed: {total_failed}")
    if total_skipped > 0:
        print(f"   ⏭️  Skipped: {total_skipped}")
    
    if dry_run_mode:
        print("\n   📋 This was a DRY RUN. No dashboards were actually deployed.")
        print("   To deploy for real, run with: --params \"dry_run_mode=false\"")
    else:
        print(f"\n   🔗 View at: {target_workspace_url}#folder{target_parent_path}")
        
        # Show deployed dashboard details from results
        if deployed_dashboards:
            print("\n   📊 Deployed dashboards:")
            for dash_result in deployed_dashboards[:10]:  # Show first 10
                dash_id = dash_result.get('steps', {}).get('dashboard', 'N/A')
                dash_name = dash_result.get('name', 'Unknown')
                print(f"      • {dash_name} → ID: {dash_id}")
    
    # Mark deployment as completed (not skipped)
    deploy_result = {'skipped': False, 'method': 'sdk_direct'}
    
else:
    print("\n⏭️ Skipping SDK deployment (deployment_method = asset_bundle)")
    # Mark deployment as skipped (will be done via CLI)
    deploy_result = {'skipped': True, 'method': 'asset_bundle'}


## Cell 5: Bundle Generation (Asset Bundle path)

In [ ]:
# ============================================================================
# BUNDLE VALIDATION (Asset Bundle path only)
# ============================================================================

if deployment_method == 'asset_bundle':
    print("\n" + "="*80)
    print("GENERATING BUNDLE STRUCTURE")
    print("="*80)
    
    print(f"\n📦 Generating bundle: {bundle_name}")
    print(f"   Output path: {bundles_path}")
    
    try:
        bundle_path = generate_bundle_structure(
            bundle_name=bundle_name,
            target_workspace_url=target_workspace_url,
            transformed_dashboards=dashboards,
            permissions_map=permissions_map,
            bundle_output_path=bundles_path,
            warehouse_id=None,
            warehouse_name=warehouse_name,
            parent_path=target_parent_path,
            embed_credentials=embed_credentials
        )
        
        print(f"\n✅ Bundle generated at: {bundle_path}")
        print("\n📋 Bundle structure:")
        print("   ├── databricks.yml")
        print("   ├── resources/")
        print("   │   └── dashboards.yml")
        print("   └── src/")
        print("       └── dashboards/")
        print(f"           └── {len(dashboards)} .lvdash.json file(s)")
        
    except Exception as e:
        print(f"\n❌ Failed to generate bundle: {e}")
        raise
    
    # Mark deployment as skipped (will be done via CLI)
    deploy_result = {'skipped': True, 'method': 'asset_bundle', 'bundle_path': bundle_path}
        
else:
    print("\n⏭️ Skipping bundle generation (deployment_method = sdk_direct)")
    bundle_path = None


## Cell 6: Deploy Bundle

In [ ]:
# ============================================================================
# NEXT STEPS
# ============================================================================

print("\n" + "="*80)
print("DEPLOYMENT COMPLETE")
print("="*80)

if deployment_method == 'sdk_direct':
    if dry_run_mode:
        print("\n🔍 DRY RUN COMPLETE")
        print("\n   No dashboards were deployed (preview mode).")
        print("\n📋 To deploy for real, run:")
        print(f"   databricks bundle run generate_deploy -t dev \\")
        print(f"     --params \"dry_run_mode=false,deployment_method=sdk_direct\" \\")
        print(f"     --profile source-workspace")
    else:
        print("\n✅ SDK DEPLOYMENT COMPLETE")
        print(f"\n   {len(deployed_dashboards)} dashboard(s) deployed to target workspace.")
        print(f"\n🔗 View dashboards at:")
        print(f"   {target_workspace_url}")
        print(f"\n📁 Location: {target_parent_path}")
        
        print("\n📋 Next steps:")
        print("   1. Verify dashboards in target workspace")
        print("   2. Clean up IP whitelist (if applicable):")
        print("      ./scripts/cleanup_ip_acl.sh --target-profile target-workspace")

elif deployment_method == 'asset_bundle':
    print("\n📦 ASSET BUNDLE READY")
    print(f"\n   Bundle saved to: {bundle_path}")
    
    if dry_run_mode:
        print("\n⚠️  NOTE: dry_run_mode applies to SDK deployment only.")
        print("   For Asset Bundle, control dry run at CLI level with:")
        print("   databricks bundle deploy --dry-run --profile target-workspace")
    
    print("\n" + "-"*80)
    print("NEXT STEPS: Run from your LOCAL TERMINAL")
    print("-"*80)
    
    print("\n📋 Option 1: Use the deploy script (recommended)")
    print(f"   # Preview first with --dry-run")
    print(f"   ./scripts/deploy_asset_bundle.sh \\")
    print(f"     --source-profile source-workspace \\")
    print(f"     --target-profile target-workspace \\")
    print(f"     --volume-base {volume_base} \\")
    print(f"     --dry-run")
    print(f"   ")
    print(f"   # Then deploy for real (remove --dry-run)")
    print(f"   ./scripts/deploy_asset_bundle.sh \\")
    print(f"     --source-profile source-workspace \\")
    print(f"     --target-profile target-workspace \\")
    print(f"     --volume-base {volume_base}")
    
    print("\n📋 Option 2: Manual CLI commands")
    print(f"   # Download bundle from volume")
    print(f"   databricks fs cp -r dbfs:{bundle_path} ./dashboard_bundle --profile source-workspace")
    print(f"   ")
    print(f"   # Deploy to target workspace (dry run first)")
    print(f"   cd ./dashboard_bundle")
    print(f"   databricks bundle deploy --dry-run --profile target-workspace  # Preview first")
    print(f"   databricks bundle deploy --profile target-workspace             # Actual deployment")
    
    print("\n" + "="*80)


## Cell 7: Verify Deployment

In [ ]:
print("\n" + "="*80)
print("VERIFYING DEPLOYMENT")
print("="*80)

# Skip verification if deployment was skipped OR if in dry run mode
if deploy_result.get('skipped'):
    print("\n⚠️  Verification skipped - deployment was not performed from notebook")
    print("\n💡 After deploying from local machine, you can verify dashboards at:")
    print(f"   {target_workspace_url}/workspace{target_parent_path}")
    verification_dashboards = []
    deployed_dashboards = []  # Empty list so schedule application can check
elif dry_run_mode:
    print("\n🔍 DRY RUN MODE - Verification skipped")
    print("\n   No dashboards were deployed (preview mode).")
    print("\n💡 To deploy and verify, run with:")
    print(f"   --params \"dry_run_mode=false,deployment_method=sdk_direct\"")
    verification_dashboards = []
    deployed_dashboards = []  # Empty list so schedule application can check
else:
    print("\n🔍 Verifying deployment results...")
    # Use deployment results from Cell 11 - NO API scan needed!
    
    # Build verification list from deployment results
    verification_dashboards = []
    for dash_result in deployed_dashboards:
        if dash_result.get('status') == 'success':
            dash_id = dash_result.get('steps', {}).get('dashboard')
            dash_name = dash_result.get('name', 'Unknown')
            if dash_id:
                verification_dashboards.append({
                    'Name': dash_name,
                    'ID': dash_id,
                    'Path': target_parent_path
                })
    
    if verification_dashboards:
        print(f"\n✅ Verified {len(verification_dashboards)} deployed dashboard(s)\n")
        df = pd.DataFrame(verification_dashboards)
        display(df)
        
        print("\n🎉 Migration complete!")
        print("\n📝 Verification checklist:")
        print("   □ Dashboards visible in target workspace")
        print("   □ Data loads correctly (new catalog/schema)")
        print("   □ Visualizations render properly")
        print("   □ Permissions applied correctly")
        print("   □ Warehouse connection works")
    else:
        print("\n⚠️  No dashboards were deployed successfully")
        print(f"   Check deployment summary above for errors")
    
    # deployed_dashboards already set from Cell 11

## Cell 8: Apply Schedules & Subscriptions

In [ ]:
print("\n" + "="*80)
print("APPLYING SCHEDULES & SUBSCRIPTIONS")
print("="*80)

# Skip schedule application if deployment was skipped OR if in dry run mode
if deploy_result.get('skipped'):
    print("\n⚠️  Schedule application skipped - deployment was not performed from notebook")
    print("\n💡 After deploying dashboards from local machine:")
    print("   1. Ensure dashboards are deployed successfully")
    print("   2. Re-run this notebook to apply schedules and subscriptions")
    print("   Or apply schedules manually via Databricks UI")
elif dry_run_mode:
    print("\n🔍 DRY RUN MODE - Schedule application skipped")
    print("\n   No schedules were applied (preview mode).")
    print("\n💡 To apply schedules for real, run with:")
    print(f"   --params \"dry_run_mode=false,deployment_method=sdk_direct\"")
elif not apply_schedules:
    print("\n⚠️  Schedule application disabled in config")
else:
    print(f"\n📅 Loading schedules from: {exported_path}")
    schedules_map = load_schedules_from_csv(exported_path)
    print(f"   ✅ Loaded schedules for {len(schedules_map)} dashboard(s)")
    
    if schedules_map:
        schedule_results = []
        
        for dash_info in deployed_dashboards:
            target_dashboard_id = dash_info['ID']
            dashboard_name = dash_info['Name']
            
            # Find schedules by ID or name
            schedules_data = schedules_map.get(target_dashboard_id) or schedules_map.get(dashboard_name)
            
            if schedules_data:
                print(f"\n📊 {dashboard_name}")
                result = apply_dashboard_schedules(
                    target_client,
                    target_dashboard_id,
                    schedules_data,
                    dry_run=schedules_dry_run
                )
                
                if result.get('status') == 'dry_run':
                    print(f"   🔍 DRY RUN - Would create: {result.get('would_create_schedules', 0)} schedules, " 
                          f"{result.get('would_create_subscriptions', 0)} subscriptions")
                elif result.get('status') == 'success':
                    print(f"   ✅ Created: {result.get('schedules_created', 0)} schedules, " 
                          f"{result.get('subscriptions_created', 0)} subscriptions")
                    if result.get('errors'):
                        print(f"   ⚠️  {len(result['errors'])} error(s) occurred")
                        for err in result['errors'][:3]:  # Show first 3 errors
                            print(f"      - {err}")
                else:
                    print(f"   ❌ Failed: {result.get('error', 'Unknown error')}")
                
                schedule_results.append(result)
        
        if not schedules_dry_run:
            total_schedules = sum(r.get('schedules_created', 0) for r in schedule_results)
            total_subs = sum(r.get('subscriptions_created', 0) for r in schedule_results)
            
            print("\n" + "="*80)
            print("SCHEDULE APPLICATION SUMMARY")
            print("="*80)
            print(f"\n✅ Total applied: {total_schedules} schedules, {total_subs} subscriptions")
            
            # Count errors
            total_errors = sum(len(r.get('errors', [])) for r in schedule_results)
            if total_errors > 0:
                print(f"⚠️  Total errors: {total_errors}")
                print("\n💡 Note: Schedule/subscription errors don't affect dashboard deployment")
        else:
            print("\n🔍 DRY RUN MODE - No schedules were created")
            print("   Set schedules_dry_run to 'false' in databricks.yml to apply schedules")
    else:
        print("\n⚠️  No schedules found to apply")
        print("   This is normal if dashboards don't have schedules in source workspace")